In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
from google.colab import files
import pandas as pd

print("Select the TWO preds files you want to compare:")
uploaded = files.upload()

csvs = [name for name in uploaded if name.endswith(".csv")]
if len(csvs) != 2:
    raise ValueError(f"Please upload exactly 2 CSV files. You uploaded: {list(uploaded)}")

FILE_A, FILE_B = csvs
a = pd.read_csv(FILE_A)
b = pd.read_csv(FILE_B)

# Short names for the output, e.g. "preds_Khushi.csv" -> "Khushi"
NAME_A = FILE_A.removeprefix("preds_").removesuffix(".csv")
NAME_B = FILE_B.removeprefix("preds_").removesuffix(".csv")

print(f"File A: {FILE_A} ({len(a)} rows)")
print(f"File B: {FILE_B} ({len(b)} rows)")

Select the TWO preds files you want to compare:


Saving preds_Prasanna Ramamurthi.csv to preds_Prasanna Ramamurthi (1).csv
Saving preds_Khushi.csv to preds_Khushi (2).csv
File A: preds_Prasanna Ramamurthi (1).csv (176 rows)
File B: preds_Khushi (2).csv (176 rows)


In [11]:
HONEST = "none"
LABELS = ["none", "preselection", "urgency", "hidden_information", "scarcity", "social_proof"]

In [12]:
import os

def clean_text(s):
    return s.astype(str).str.strip().str.lower()

def clean_image(s):
    return s.astype(str).str.strip().str.replace("\\", "/").map(os.path.basename).str.lower()

pa = a[["image", "label", "predicted"]].rename(columns={"label": "true", "predicted": "a_pred"})
pb = b[["image", "label", "predicted"]].rename(columns={"label": "b_label", "predicted": "b_pred"})
for t in (pa, pb):
    t["image"] = clean_image(t["image"])

df = pa.merge(pb, on="image")
for col in ["true", "b_label", "a_pred", "b_pred"]:
    df[col] = clean_text(df[col])

print(f"Matched {len(df)} screenshots")
print(f"  {NAME_A}: {len(pa)} | {NAME_B}: {len(pb)}")

only_a = set(pa["image"]) - set(pb["image"])
only_b = set(pb["image"]) - set(pa["image"])
if only_a: print(f"  {len(only_a)} images only in {NAME_A}: {sorted(only_a)[:5]}")
if only_b: print(f"  {len(only_b)} images only in {NAME_B}: {sorted(only_b)[:5]}")

mismatch = df[df["true"] != df["b_label"]]
if len(mismatch):
    print(f"\nWARNING: {len(mismatch)} images have different true labels in the two files.")
    print(f"Using {NAME_A}'s labels. Examples:")
    display(mismatch[["image", "true", "b_label"]].head(10))
else:
    print("  Both files agree on every true label.")

df = df.drop(columns="b_label")

Matched 176 screenshots
  Prasanna Ramamurthi (1): 176 | Khushi (2): 176
  Both files agree on every true label.


In [13]:
for col in ["a_pred", "b_pred"]:
    df.loc[~df[col].isin(LABELS), col] = "PARSE_FAIL"

df["a_ok"] = df["a_pred"] == df["true"]
df["b_ok"] = df["b_pred"] == df["true"]

print(f"{NAME_A} accuracy: {df['a_ok'].mean():.1%}")
print(f"{NAME_B} accuracy: {df['b_ok'].mean():.1%}")

Prasanna Ramamurthi (1) accuracy: 63.6%
Khushi (2) accuracy: 75.0%


In [14]:
counts = pd.DataFrame({
    "true": df["true"].value_counts(),
    f"{NAME_A} predicted": df["a_pred"].value_counts(),
    f"{NAME_B} predicted": df["b_pred"].value_counts(),
}).reindex(LABELS + ["PARSE_FAIL"]).fillna(0).astype(int)
display(counts)

,true,Prasanna Ramamurthi (1) predicted,Khushi (2) predicted
none,95,94,107
preselection,39,42,41
urgency,17,14,13
hidden_information,14,20,13
scarcity,9,1,1
social_proof,2,5,1
PARSE_FAIL,0,0,0


In [15]:
from sklearn.metrics import f1_score

cats = [c for c in LABELS if c in set(df["true"])]
scores = pd.DataFrame({
    "screenshots": df["true"].value_counts().reindex(cats),
    f"{NAME_A} F1": f1_score(df["true"], df["a_pred"], labels=cats, average=None, zero_division=0),
    f"{NAME_B} F1": f1_score(df["true"], df["b_pred"], labels=cats, average=None, zero_division=0),
}, index=cats)
scores.loc["MACRO AVG"] = [len(df), scores[f"{NAME_A} F1"].mean(), scores[f"{NAME_B} F1"].mean()]
scores["screenshots"] = scores["screenshots"].astype(int)
display(scores.round(3))

,screenshots,Prasanna Ramamurthi (1) F1,Khushi (2) F1
none,95,0.751,0.822
preselection,39,0.691,0.725
urgency,17,0.452,0.733
hidden_information,14,0.294,0.593
scarcity,9,0.200,0.200
social_proof,2,0.000,0.000
MACRO AVG,176,0.398,0.512


In [16]:
def bucket(row):
    if row.a_ok and row.b_ok: return "both right"
    if row.a_ok: return f"only {NAME_A}"
    if row.b_ok: return f"only {NAME_B}"
    return "both wrong"

df["who"] = df.apply(bucket, axis=1)
order = ["both right", f"only {NAME_A}", f"only {NAME_B}", "both wrong"]
who = pd.crosstab(df["true"], df["who"]).reindex(index=cats, columns=order, fill_value=0)
who.loc["OVERALL"] = who.sum()
who["total"] = who.sum(axis=1)
display(who)

who,both right,only Prasanna Ramamurthi (1),only Khushi (2),both wrong,total
true,,,,,
none,66,5,17,7,95
preselection,24,4,5,6,39
urgency,6,1,5,5,17
hidden_information,3,2,5,4,14
scarcity,1,0,0,8,9
social_proof,0,0,0,2,2
OVERALL,100,12,32,32,176


In [17]:
same = (df["a_pred"] == df["b_pred"]) & (df["a_pred"] != "PARSE_FAIL")
print(f"Same answer: {same.sum()} / {len(df)} ({same.mean():.1%})")

Same answer: 124 / 176 (70.5%)


In [18]:
honest = df[df["true"] == HONEST]
print(f"Honest screenshots: {len(honest)}")
if len(honest) != 46:
    print("  Note: the task expected 46 honest screenshots.")

for col, name in [("a_pred", NAME_A), ("b_pred", NAME_B)]:
    dark = (~honest[col].isin([HONEST, "PARSE_FAIL"])).sum()
    print(f"{name}: {dark} / {len(honest)} wrongly called dark ({dark/len(honest):.1%})")

Honest screenshots: 95
  Note: the task expected 46 honest screenshots.
Prasanna Ramamurthi (1): 24 / 95 wrongly called dark (25.3%)
Khushi (2): 12 / 95 wrongly called dark (12.6%)


In [19]:
for col, name in [("a_pred", NAME_A), ("b_pred", NAME_B)]:
    fails = df.loc[df[col] == "PARSE_FAIL", "image"].tolist()
    print(f"{name}: {len(fails)} parse fails")
    for img in fails:
        print(f"   {img}")

Prasanna Ramamurthi (1): 0 parse fails
Khushi (2): 0 parse fails


In [20]:
dis = df.loc[df["a_pred"] != df["b_pred"], ["image", "true", "a_pred", "b_pred"]]
dis = dis.rename(columns={"true": "true label", "a_pred": NAME_A, "b_pred": NAME_B})

print(f"{len(dis)} screenshots where the two files disagree")
display(dis.reset_index(drop=True))

out = f"disagreements_{NAME_A}_vs_{NAME_B}.csv"
dis.to_csv(out, index=False)
files.download(out)
print(f"Downloaded {out}")

52 screenshots where the two files disagree


,image,true label,Prasanna Ramamurthi (1),Khushi (2)
0,12810_9mifsrp.jpg,preselection,hidden_information,none
1,1098.jpg,none,preselection,none
2,music_21--tunein-radio-0-16_e31e.jpg,preselection,hidden_information,preselection
3,us.pandora.net.png,none,urgency,none
4,1052.jpg,none,none,hidden_information
5,www.gasbike.net.png,urgency,none,urgency
6,1366_8h8kxdb.jpg,preselection,hidden_information,preselection
7,13686.jpg,preselection,urgency,none
8,2ndsworld.com.au.png,none,social_proof,none
9,13413_vr8is1f.jpg,hidden_information,none,hidden_information


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded disagreements_Prasanna Ramamurthi (1)_vs_Khushi (2).csv
